In [89]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import numpy as np
from sklearn.impute import KNNImputer
import os 

In [90]:
def outlier_removal(df, column):
    """
    Remove outliers de uma coluna numérica de um DataFrame usando limiares automáticos.
    
    Parâmetros:
    df (pd.DataFrame): DataFrame de entrada.
    column (str): Nome da coluna a ser tratada.
    
    Retorna:
    pd.DataFrame: DataFrame com os outliers substituídos por NaN.
    """
    # Substitui valores inválidos (-1) por NaN
    df = df.copy()
    df[column] = df[column].replace(-1, np.nan)

    # Converte os valores não nulos em array NumPy
    r = df[column].dropna().to_numpy()

    # Verifica se há dados suficientes para análise
    if r.size == 0:
        print("Coluna não contém valores suficientes para análise.")
        return df

    # Normaliza os dados dividindo pelo valor máximo
    r_max = np.max(r)
    if r_max == 0:
        print("Valor máximo é zero, normalização inválida.")
        return df
    r_norm = r / r_max

    # Estima limiar inferior baseado na maior variação percentual nos percentis baixos
    p_min = np.linspace(0.1, 2, 20)
    perc_min = [np.percentile(r_norm, i) for i in p_min]
    diff_perc_min = np.diff(perc_min)
    index_min = np.argmax(diff_perc_min)
    thres_min = np.mean(perc_min[index_min:index_min + 2])

    # Estima limiar superior baseado na maior variação percentual nos percentis altos
    p_max = np.linspace(98, 100, 20)
    perc_max = [np.percentile(r_norm, i) for i in p_max]
    diff_perc_max = np.diff(perc_max)
    index_max = np.argmax(diff_perc_max)
    thres_max = np.mean(perc_max[index_max:index_max + 2])

    # Marca como NaN os valores fora dos limiares
    r_filtered_norm = np.where(
        (r_norm < thres_min) | (r_norm > thres_max),
        np.nan,
        r_norm
    )

    # Desfaz a normalização
    r_filtered = r_filtered_norm * r_max

    # Cria cópia do DataFrame e substitui apenas os valores filtrados
    df_filtered = df.copy()
    # Substitui apenas as posições não nulas originais pelos valores filtrados
    non_na_indices = df[column].dropna().index
    df_filtered.loc[non_na_indices, column] = r_filtered

    return df_filtered

In [91]:
def impute_knn(df, k=5):
    """
    Imputa valores ausentes na coluna 'Vazao' usando KNN Imputer.
    
    Parâmetros:
    df (pd.DataFrame): DataFrame de entrada.
    k (int): Número de vizinhos para o KNN.
    
    Retorna:
    pd.DataFrame: DataFrame com valores imputados.
    """
    imputer = KNNImputer(n_neighbors=k)
    df_copy = df.copy()
    df_copy[['Vazao']] = imputer.fit_transform(df_copy[['Vazao']])
    return df_copy

def impute_rolling_median(df, window_size=3):
    """
    Imputa valores ausentes na coluna 'Vazao' usando mediana móvel.
    Caso ainda restem NaNs, usa a mediana global.
    
    Parâmetros:
    df (pd.DataFrame): DataFrame de entrada.
    window_size (int): Tamanho da janela de cálculo da mediana.
    
    Retorna:
    pd.DataFrame: DataFrame com valores imputados.
    """
    df_copy = df.copy()
    rolling_median = df_copy['Vazao'].rolling(
        window=window_size, min_periods=1
    ).median()
    df_copy['Vazao'] = df_copy['Vazao'].fillna(rolling_median)

    # Se ainda restarem NaNs, preenche com a mediana global
    global_median = df_copy['Vazao'].median()
    df_copy['Vazao'] = df_copy['Vazao'].fillna(global_median)

    return df_copy

def impute_rolling_average(df, window_size=3):
    """
    Imputa valores ausentes na coluna 'vazao' usando média móvel.
    Caso ainda restem NaNs, usa a média global.
    
    Parâmetros:
    df (pd.DataFrame): DataFrame de entrada.
    window_size (int): Tamanho da janela de cálculo da média.
    
    Retorna:
    pd.DataFrame: DataFrame com valores imputados.
    """
    df_copy = df.copy()
    rolling_mean = df_copy['vazao'].rolling(
        window=window_size, min_periods=1
    ).mean()
    df_copy['vazao'] = df_copy['vazao'].fillna(rolling_mean)

    # Se ainda restarem NaNs, preenche com a média global
    global_mean = df_copy['vazao'].mean()
    df_copy['vazao'] = df_copy['vazao'].fillna(global_mean)

    return df_copy

def linear_interpolation(df, limit_direction='both', method='linear'):
    """
    Imputa valores ausentes usando interpolação linear.
    
    Parâmetros:
    df (pd.DataFrame): DataFrame de entrada.
    limit_direction (str): Direção para interpolar ('forward', 'backward', 'both').
    method (str): Método de interpolação (default: 'linear').
    
    Retorna:
    pd.DataFrame: DataFrame com valores imputados.
    """
    df_copy = df.copy()
    df_copy['vazao'] = df_copy['vazao'].interpolate(
        method=method, limit_direction=limit_direction
    )
    return df_copy

In [92]:
def decomposicao_svd(df):
    """
    Executa a decomposição SVD em uma matriz (DataFrame).
    
    Retorna:
    U, S, Vt : componentes da decomposição.
    """
    U, S, Vt = np.linalg.svd(df, full_matrices=True)
    return U, S, Vt

def grafico_variabilidade(variabilidade, S):
    """
    Plota o gráfico da variabilidade acumulada a partir dos valores singulares.
    
    Parâmetros:
    variabilidade : lista ou array com a variabilidade acumulada.
    S : array de valores singulares.
    """
    plt.plot(
        range(1, len(variabilidade) + 1), 
        variabilidade, 
        marker='o', 
        markersize=3,
        markerfacecolor='teal', 
        markeredgecolor='teal', 
        color='darkturquoise'
    )
    plt.xlabel('Número de Valores Singulares')
    plt.ylabel('Variabilidade Acumulada')
    plt.title('Valores Singulares por Variabilidade Acumulada')
    plt.grid(color='lightgray', alpha=0.7)
    plt.show()

def componentes_principais(r, U, S, Vt):
    """
    Retorna as componentes principais truncadas com base no rank r.
    
    Parâmetros:
    r : int, número de componentes principais a manter.
    U, S, Vt : componentes da decomposição SVD.
    
    Retorna:
    U_reduced, S_reduced, Vt_reduced
    """
    U_reduced = U[:, :r]
    S_reduced = S[:r]
    Vt_reduced = Vt[:r, :]
    return U_reduced, S_reduced, Vt_reduced

def matriztodf(matriz_reconstruida):
    """
    Converte uma matriz reconstruída via SVD em um DataFrame no formato original.
    Faz reshape e reordena para vetorizar novamente.
    
    Parâmetros:
    matriz_reconstruida : ndarray ou DataFrame
    
    Retorna:
    df : DataFrame com coluna 'vazao'
    """
    matriz = np.array(matriz_reconstruida)
    linhas, colunas = matriz.shape
    vetor = matriz.T.reshape(-1)  # Desfaz o reshape original
    df = pd.DataFrame(vetor, columns=['Vazao'])
    return df

def calcular_rmse(df1, df2, coluna):
    """
    Calcula o RMSE entre os valores imputados e os valores originais.
    
    Parâmetros:
    df1, df2 : DataFrames a comparar.
    coluna : str, nome da coluna a usar.
    
    Retorna:
    rmse : float
    """
    indices_comuns = df1.index.intersection(df2.index)
    valores_df1 = df1.loc[indices_comuns, coluna]
    valores_df2 = df2.loc[indices_comuns, coluna]
    rmse = np.sqrt(np.mean((valores_df1 - valores_df2) ** 2))
    return rmse

def gerar_arq_csv(df_base, df_svd, caminho_base, nome_arquivo_csv):
    """
    Gera arquivo CSV a partir da coluna reconstruída pelo SVD.
    Mantém o DataFrame base para preservar colunas extras (ex: datetime).
    
    Parâmetros:
    df_base : DataFrame base com estrutura original.
    df_svd : DataFrame com coluna imputada 'vazao'.
    caminho_base : str, caminho raiz onde salvar.
    nome_arquivo_csv : str, nome do arquivo.
    
    Retorna:
    DataFrame final salvo.
    """
    df_final = df_base.copy()
    df_final['Vazao'] = df_svd['Vazao']

    # Remove linhas sem imputação (NaN)
    df_final = df_final.dropna(subset=['Vazao'])

    # Cria pasta, se não existir
    caminho_svd = os.path.join(caminho_base, 'svd')
    os.makedirs(caminho_svd, exist_ok=True)

    caminho_arquivo_csv = os.path.join(caminho_svd, nome_arquivo_csv)
    df_final.to_csv(caminho_arquivo_csv, index=False)

    print(f"Arquivo CSV '{caminho_arquivo_csv}' gerado com sucesso!")
    return df_final

def matriz(path):
    """
    Lê CSV, remove outliers, cria matriz para SVD, interpolação e máscara de NaNs.
    A matriz é reshapeada em blocos de 28 elementos por coluna.
    
    Retorna:
    matriz_original, matriz_mascara, matriz_interpolada, df_datetime
    """
    df = pd.read_csv(path)

    # Remove outliers antes da imputação
    df = outlier_removal(df, 'Vazao')

    # Guarda datetime ou outras colunas
    df_datetime = df.copy()
    df_datetime.drop(columns=['Vazao'], inplace=True)

    # Normaliza valores inválidos
    df['Vazao'] = df['Vazao'].replace(-1, np.nan)

    # Vetor de dados
    vazao = df['Vazao'].values

    # Define número de colunas fixo: blocos de 28 (ex: dias, semanas)
    num_dados = len(vazao)
    num_colunas = num_dados // 28

    matriz = vazao[:num_colunas * 28].reshape(num_colunas, 28).T
    matriz_original = pd.DataFrame(matriz)

    # Gera matriz interpolada para reconstrução
    df_interpolado = df['Vazao'].interpolate(method='linear', limit_direction='both')
    vazao_interp = df_interpolado.values
    matriz_interpolada = vazao_interp[:num_colunas * 28].reshape(num_colunas, 28).T
    matriz_interpolada = pd.DataFrame(matriz_interpolada)

    # Cria máscara de NaNs para controle posterior
    mask = np.isnan(matriz)
    matriz_mascara = pd.DataFrame(mask)

    return matriz_original, matriz_mascara, matriz_interpolada, df_datetime

In [93]:
def get_dataset_path_list(directory):
    """
    Retorna uma lista com os caminhos absolutos de todos os arquivos
    presentes em um diretório.

    Parâmetros:
    directory (str): Caminho do diretório a ser listado.

    Retorna:
    list: Lista de caminhos completos dos arquivos.
    """
    dataset_path_list = []

    # Verifica se o diretório existe
    if not os.path.isdir(directory):
        raise ValueError(f"O diretório '{directory}' não existe ou não é um diretório válido.")

    # Percorre os arquivos do diretório
    for file in os.listdir(directory):
        path = os.path.join(directory, file)
        # Garante que seja um arquivo regular (não uma pasta)
        if os.path.isfile(path):
            dataset_path_list.append(path)

    return dataset_path_list

In [94]:
def apply_basic_imputations(source_path, destination_path, csv_path_list):
    """
    Aplica técnicas básicas de imputação (KNN, mediana móvel, média móvel, interpolação linear)
    em uma lista de arquivos CSV, salvando os resultados em pastas separadas.

    Parâmetros:
    - source_path (str): Caminho base dos arquivos de origem.
    - destination_path (str): Caminho base para salvar os resultados.
    - csv_path_list (list): Lista de caminhos CSV a processar.
    """
    for caminho in csv_path_list:
        df = pd.read_csv(caminho)

        if df.shape[0] < 28:
            print(f"Arquivo '{caminho}' ignorado: número de linhas insuficiente (< 28) para imputação.")
            continue

        df = outlier_removal(df, 'vazao')

        df_knn = impute_knn(df.copy())
        df_rolling_median = impute_rolling_median(df.copy())
        df_rolling_average = impute_rolling_average(df.copy())
        df_interpolation = linear_interpolation(df.copy())

        # Cria as subpastas das técnicas, se não existirem
        techniques = ['knn', 'mediana-movel', 'media-movel', 'interpolacao-linear']
        for technique in techniques:
            technique_dir = os.path.join(destination_path, technique)
            os.makedirs(technique_dir, exist_ok=True)

        # Define caminhos de saída relativos à estrutura original
        relative_path = os.path.relpath(caminho, source_path)
        output_knn = os.path.join(destination_path, 'knn', relative_path)
        output_median = os.path.join(destination_path, 'mediana-movel', relative_path)
        output_average = os.path.join(destination_path, 'media-movel', relative_path)
        output_interpolation = os.path.join(destination_path, 'interpolacao-linear', relative_path)

        # Cria subpastas intermediárias, se necessário
        os.makedirs(os.path.dirname(output_knn), exist_ok=True)
        os.makedirs(os.path.dirname(output_median), exist_ok=True)
        os.makedirs(os.path.dirname(output_average), exist_ok=True)
        os.makedirs(os.path.dirname(output_interpolation), exist_ok=True)

        # Salva os arquivos CSV
        df_knn.to_csv(output_knn, index=False)
        df_rolling_median.to_csv(output_median, index=False)
        df_rolling_average.to_csv(output_average, index=False)
        df_interpolation.to_csv(output_interpolation, index=False)

        print(f"Arquivo '{caminho}' processado com imputações básicas e salvo.")

def apply_svd_imputation(destination_path, csv_path_list):
    """
    Aplica imputação iterativa baseada em SVD para cada arquivo na lista.
    Salva o resultado final reconstruído no diretório de destino.

    Parâmetros:
    - destination_path (str): Caminho base para salvar resultados.
    - csv_path_list (list): Lista de caminhos CSV a processar.
    """
    resultados = {}

    for caminho_csv in csv_path_list:
        df = pd.read_csv(caminho_csv)

        if df.shape[0] < 28:
            print(f"Arquivo '{caminho_csv}' ignorado: número de linhas insuficiente (< 28) para SVD.")
            continue

        nome_arquivo = os.path.basename(caminho_csv)
        resultados[nome_arquivo] = {'interpolacao_linear': None, 'svd_final': None}

        # Cria matriz base, máscara e interpolação inicial
        matriz_original, matriz_mascara, matriz_interpolada, df_datetime = matriz(caminho_csv)
        resultados[nome_arquivo]['interpolacao_linear'] = matriz_interpolada.copy()

        # Itera até convergência (RMSE) ou limite de iterações
        A_anterior = matriz_interpolada.values.copy()
        rmse = float('inf')
        max_iter = 300
        n_iter = 0

        while rmse >= 1e-3 and n_iter <= max_iter:
            U, S, Vt = decomposicao_svd(matriz_interpolada)
            variabilidade = np.cumsum(S**2) / np.sum(S**2)

            # Seleciona r com base em 95% de variabilidade
            r = np.where(variabilidade >= 0.95)[0][0] + 1

            U_reduzido, S_reduzido, Vt_reduzido = componentes_principais(r, U, S, Vt)
            S_reduzido_matriz = np.diag(S_reduzido)

            A_aproximada = np.dot(U_reduzido @ S_reduzido_matriz, Vt_reduzido)

            # Preenche apenas posições ausentes com aproximação SVD
            matriz_interpolada = matriz_original.fillna(pd.DataFrame(A_aproximada))

            resultados[nome_arquivo]['svd_final'] = matriz_interpolada.copy()

            # Calcula RMSE entre iterações
            rmse = np.sqrt(np.mean((A_aproximada - A_anterior) ** 2))
            A_anterior = A_aproximada.copy()
            n_iter += 1

        print(f"Arquivo '{caminho_csv}' processado via SVD: RMSE final = {rmse:.6f} em {n_iter} iterações.")

        # Reconstrói formato vetorial final
        df_svd = matriztodf(resultados[nome_arquivo]["svd_final"])

        # Salva arquivo CSV final com datetime original
        gerar_arq_csv(df_datetime, df_svd, destination_path, nome_arquivo)

In [95]:
def apply_all_imputations(source_path, destination_path):
    """
    Executa o fluxo completo de imputações em todos os arquivos CSV de um diretório.
    
    Parâmetros:
    - source_path (str): Caminho do diretório com os arquivos de origem.
    - destination_path (str): Caminho base onde os resultados serão salvos.
    
    Fluxo:
    1) Gera lista de arquivos CSV.
    2) Executa imputações básicas (KNN, Rolling, Interpolação Linear). [Opcional]
    3) Executa imputação avançada via SVD.
    """
    # Obtém lista de caminhos dos arquivos CSV no diretório de origem
    csv_path_list = get_dataset_path_list(source_path)

    if not csv_path_list:
        print(f"Nenhum arquivo CSV encontrado no diretório '{source_path}'. Nada a processar.")
        return

    print(f"{len(csv_path_list)} arquivos encontrados em '{source_path}'. Iniciando processamento...")

    # Descomente a linha abaixo para aplicar imputações básicas também
    # apply_basic_imputations(source_path, destination_path, csv_path_list)

    # Executa imputação por SVD (sempre ativa)
    apply_svd_imputation(destination_path, csv_path_list)

    print("Processamento concluído.")

In [96]:
# Caminho de origem: onde estão os arquivos CSV originais para imputação
source_path = '../datasets/choosen-best-svd/'

# Caminho de destino: onde serão salvos os arquivos CSV com valores imputados
destination_path = '../datasets/svd-imputed-choosen-best-svd/'


In [ ]:
# Executa o pipeline completo de imputações:
# - Busca todos os arquivos CSV no diretório definido em `source_path`
# - Remove outliers, aplica imputações básicas (KNN, Rolling, Interpolação Linear)
# - Executa a imputação avançada via SVD iterativo
# - Salva cada resultado em subpastas dentro de `destination_path`
apply_all_imputations(source_path, destination_path)


In [ ]:
def gabriel_eigen_impute(X, tol=1e-4, max_iter=100, m=None):
    """
    Imputa valores ausentes em uma matriz usando o método de Gabriel & Eigen,
    com SVD regularizado e atualização iterativa.

    Parâmetros:
    - X : matriz com NaNs
    - tol : tolerância de convergência
    - max_iter : número máximo de iterações
    - m : rank desejado (se None, detecta automaticamente)

    Retorna:
    - X imputada
    """
    X = np.array(X, dtype=float)
    n, p = X.shape

    # Substitui NaNs pela média de cada coluna
    col_means = np.nanmean(X, axis=0)
    inds = np.where(np.isnan(X))
    X[inds] = np.take(col_means, inds[1])

    X_prev = np.copy(X)
    converged = False
    iter_count = 0

    while not converged and iter_count < max_iter:
        # Padroniza colunas
        col_means = np.mean(X, axis=0)
        col_stds = np.std(X, axis=0, ddof=1)
        col_stds[col_stds == 0] = 1  # Evita divisão por zero
        X_std = (X - col_means) / col_stds

        # Executa SVD regularizado
        U, D, Vt = reg_svd(X_std, m)

        # Reconstrói matriz padronizada
        X_hat_std = U @ np.diag(D) @ Vt

        # Reverte padronização
        X_hat = X_hat_std * col_stds + col_means

        # Atualiza apenas posições ausentes
        X[inds] = X_hat[inds]

        # Verifica convergência
        diff = np.linalg.norm(X - X_prev) / np.linalg.norm(X_prev)
        if diff < tol:
            converged = True

        X_prev = np.copy(X)
        iter_count += 1

    print(f"Convergiu após {iter_count} iterações (RMSE relativo: {diff:.6e})")

    return X

def reg_svd(X_std, m=None):
    """
    Executa SVD regularizado para uma matriz padronizada.

    Parâmetros:
    - X_std : matriz padronizada
    - m : rank desejado (opcional)

    Retorna:
    - U_final, D, Vt_final : componentes SVD truncadas
    """
    n, p = X_std.shape
    k = min(n, p)

    lambdas = np.arange(0, 1.1, 0.1)
    min_f = np.inf
    best_lambda = 0
    best_U = None
    best_V = None

    # Busca o lambda ótimo
    for lmbda in lambdas:
        U, V, f = compute_reg_svd(X_std, k, lmbda)
        if f < min_f:
            min_f = f
            best_lambda = lmbda
            best_U = U
            best_V = V

    UVt = best_U @ best_V.T
    U_final, D, Vt_final = np.linalg.svd(UVt, full_matrices=False)

    if m is None:
        cumulative_energy = np.cumsum(D**2)
        total_energy = cumulative_energy[-1]
        m = np.searchsorted(cumulative_energy, 0.8 * total_energy) + 1  # 80% da energia

    U_final = U_final[:, :m]
    D = D[:m]
    Vt_final = Vt_final[:m, :]

    print(f"Lambda ótimo encontrado: {best_lambda:.2f} | Rank final: {m}")

    return U_final, D, Vt_final

def compute_reg_svd(X, k, lmbda, tol=1e-4, max_iter=100):
    """
    Computa o SVD regularizado via iteração alternada.

    Parâmetros:
    - X : matriz padronizada
    - k : número de componentes
    - lmbda : parâmetro de regularização

    Retorna:
    - U, V, f : matrizes fatoradas e valor da função objetivo
    """
    n, p = X.shape
    V = np.random.rand(p, k)
    U = np.zeros((n, k))
    f_prev = np.inf

    I_k = np.eye(k)

    for _ in range(max_iter):
        VVt_inv = np.linalg.pinv(V.T @ V + lmbda * I_k)
        U = X @ V @ VVt_inv

        UUt_inv = np.linalg.pinv(U.T @ U + lmbda * I_k)
        V = X.T @ U @ UUt_inv

        UVt = U @ V.T
        residual = X - UVt
        f = np.linalg.norm(residual, 'fro')**2 + lmbda * (np.linalg.norm(U, 'fro')**2 + np.linalg.norm(V, 'fro')**2)

        if abs(f_prev - f) < tol:
            break
        f_prev = f

    return U, V, f

def df_to_matrix_transformation(df):
    """
    Converte DataFrame em matriz NumPy, substituindo -1 por NaN.

    Parâmetros:
    - df : DataFrame com coluna 'vazao'

    Retorna:
    - matriz NumPy
    """
    df['vazao'] = df['vazao'].replace(-1, np.nan)
    matrix = df.to_numpy()
    return matrix

In [ ]:
def gabriel_eigen_impute_df(df, tol=1e-4, max_iter=100, m=None):
    """
    Aplica o método de imputação Gabriel Eigen Impute em um DataFrame,
    mantendo a coluna 'Timestamp' intacta.

    Parâmetros:
    - df : DataFrame com coluna 'Timestamp' e colunas numéricas com possíveis NaNs
    - tol : tolerância de convergência para iterações
    - max_iter : número máximo de iterações
    - m : rank desejado (se None, é determinado automaticamente)

    Retorna:
    - DataFrame com valores imputados e 'Timestamp' original
    """

    # Separa a coluna de timestamp das colunas numéricas
    timestamp_col = df['Timestamp'].reset_index(drop=True)
    value_cols = df.drop(columns=['Timestamp'])

    # Converte valores para array NumPy para processamento SVD
    X = value_cols.to_numpy(dtype=float)

    # Substitui NaNs pela média da coluna (valor inicial)
    col_means = np.nanmean(X, axis=0)
    inds = np.where(np.isnan(X))
    X[inds] = np.take(col_means, inds[1])

    # Variáveis para controle de convergência
    X_prev = np.copy(X)
    converged = False
    iter_count = 0

    while not converged and iter_count < max_iter:
        # Padroniza colunas (zero média, variância unitária)
        col_means = np.mean(X, axis=0)
        col_stds = np.std(X, axis=0, ddof=1)
        col_stds[col_stds == 0] = 1  # Evita divisão por zero
        X_std = (X - col_means) / col_stds

        # Executa SVD regularizado
        U, D, Vt = reg_svd(X_std, m)

        # Reconstrói matriz padronizada
        X_hat_std = U @ np.diag(D) @ Vt

        # Reverte padronização
        X_hat = X_hat_std * col_stds + col_means

        # Atualiza apenas os NaNs originais
        X[inds] = X_hat[inds]

        # Verifica convergência
        diff = np.linalg.norm(X - X_prev) / np.linalg.norm(X_prev)
        if diff < tol:
            converged = True

        X_prev = np.copy(X)
        iter_count += 1

    print(f"Convergiu após {iter_count} iterações | Diferença final: {diff:.6e}")

    # Cria novo DataFrame com valores imputados e timestamp original
    df_imputed = pd.DataFrame(X, columns=value_cols.columns)
    df_imputed.insert(0, 'Timestamp', timestamp_col)  # Garante ordem

    return df_imputed


In [ ]:
def apply_reg_svd_imputation(source_path, destination_path, csv_path_list):
    """
    Executa imputação de valores ausentes usando Gabriel Eigen Impute (SVD regularizado)
    em todos os arquivos CSV de uma lista.

    Para cada arquivo:
    - Remove outliers
    - Aplica imputação SVD regularizada
    - Salva o resultado em uma subpasta 'reg_svd' dentro do destino

    Parâmetros:
    - source_path (str): Diretório base de entrada
    - destination_path (str): Diretório base de saída
    - csv_path_list (list): Lista de caminhos CSV a processar
    """
    for caminho in csv_path_list:
        df = pd.read_csv(caminho)

        # Remove outliers na coluna 'vazao'
        df = outlier_removal(df, 'vazao')

        # Aplica imputação Gabriel Eigen SVD
        df_reg = gabriel_eigen_impute_df(df.copy())

        # Define subpasta para técnica usada
        technique = 'reg_svd'
        technique_dir = os.path.join(destination_path, technique)
        os.makedirs(technique_dir, exist_ok=True)

        # Cria estrutura de subpastas mantendo o caminho relativo
        relative_path = os.path.relpath(caminho, source_path)
        output_reg = os.path.join(technique_dir, relative_path)

        # Garante que a pasta final existe
        os.makedirs(os.path.dirname(output_reg), exist_ok=True)

        # Salva arquivo imputado
        df_reg.to_csv(output_reg, index=False)

        print(f"Processado e salvo: '{output_reg}' (original: '{caminho}')")


In [ ]:
def weighted_average(window):
    """
    Calcula a média ponderada de uma janela,
    ignorando valores NaN.

    Pesos: 1, 2, 3, ..., n (mais peso para os valores mais recentes).

    Parâmetros:
    - window : sequência numérica (pode conter NaN)

    Retorna:
    - valor da média ponderada ou NaN se todos os valores forem NaN
    """
    window = np.array(window)
    mask = ~np.isnan(window)

    if np.sum(mask) == 0:
        return np.nan

    weights = np.arange(1, np.sum(mask) + 1)
    return np.average(window[mask], weights=weights)

def weighted_rolling_mean(df, window_size=30):
    """
    Aplica uma média móvel ponderada na coluna 'Vazao',
    preenchendo valores ausentes com a média calculada.

    Parâmetros:
    - df : DataFrame de entrada, deve conter a coluna 'Vazao'
    - window_size : tamanho da janela (default: 30)

    Retorna:
    - DataFrame com a coluna 'Vazao' imputada.
    """
    df_copy = df.copy()

    # Substitui -1 por NaN (se ainda não tiver sido tratado)
    df_copy['Vazao'] = df_copy['Vazao'].replace(-1, np.nan)

    # Calcula a média móvel ponderada
    weighted_rolling = df_copy['Vazao'].rolling(
        window=window_size, min_periods=1
    ).apply(weighted_average, raw=False)

    # Substitui valores ausentes pela média ponderada
    df_copy['Vazao'] = df_copy['Vazao'].fillna(weighted_rolling)

    # Preenche valores iniciais restantes (caso ainda tenha NaNs no começo)
    df_copy['Vazao'] = df_copy['Vazao'].bfill()

    return df_copy


In [ ]:
def apply_weighted_rolling_mean(source_path, destination_path, csv_path_list):
    """
    Aplica imputação por média móvel ponderada em uma lista de arquivos CSV.
    - Remove outliers
    - Imputa com média móvel ponderada
    - Salva em subpasta 'weighted_rolling_mean'

    Parâmetros:
    - source_path : caminho base de entrada
    - destination_path : caminho base de saída
    - csv_path_list : lista de caminhos CSV
    """
    for caminho in csv_path_list:
        df = pd.read_csv(caminho)

        # (Opcional) Ver parte inicial do DataFrame original
        # print(df.head(15))

        # Remove outliers na coluna 'vazao'
        df = outlier_removal(df, 'vazao')

        # Aplica média móvel ponderada
        df_weighted = weighted_rolling_mean(df.copy())

        # (Opcional) Ver parte inicial após imputação
        # print(df_weighted.head(15))

        # Define subpasta para técnica
        technique = 'weighted_rolling_mean'
        technique_dir = os.path.join(destination_path, technique)
        os.makedirs(technique_dir, exist_ok=True)

        # Cria caminho relativo de saída mantendo estrutura de pastas original
        relative_path = os.path.relpath(caminho, source_path)
        output_path = os.path.join(technique_dir, relative_path)

        # Garante que as subpastas existem
        os.makedirs(os.path.dirname(output_path), exist_ok=True)

        # Salva arquivo imputado
        df_weighted.to_csv(output_path, index=False)

        print(f"Processado e salvo: '{output_path}' (original: '{caminho}')")

In [98]:
source_path_bbr = r'C:\Users\janaina\OneDrive\Documentos\UECE-RNP-2024-REF\intervalos-vazao\bbr'
source_path_cubic = r'C:\Users\janaina\OneDrive\Documentos\UECE-RNP-2024-REF\intervalos-vazao\cubic'
destination_path_bbr = r'C:\Users\janaina\OneDrive\Documentos\UECE-RNP-2024-REF\intervalos-vazao\bbr-imputed'
destination_path_cubic = r'C:\Users\janaina\OneDrive\Documentos\UECE-RNP-2024-REF\intervalos-vazao\cubic-imputed'
csv_path_list = get_dataset_path_list(source_path_bbr) + get_dataset_path_list(source_path_cubic)
print(f"{len(csv_path_list)} arquivos encontrados para processamento.")

def safe_apply_svd_imputation(destination_path, csv_path_list):
	"""
	Aplica SVD apenas em arquivos que possuem a coluna 'Vazao'.
	"""
	valid_csvs = []
	for path in csv_path_list:
		try:
			df = pd.read_csv(path, nrows=1)
			if 'Vazao' in df.columns:
				valid_csvs.append(path)
			else:
				print(f"Aviso: '{path}' ignorado (coluna 'Vazao' ausente).")
		except Exception as e:
			print(f"Erro ao ler '{path}': {e}")

	if valid_csvs:
		apply_svd_imputation(destination_path, valid_csvs)
	else:
		print("Nenhum arquivo válido para imputação SVD.")

safe_apply_svd_imputation(destination_path_bbr, csv_path_list)
safe_apply_svd_imputation(destination_path_cubic, csv_path_list)



146 arquivos encontrados para processamento.
Arquivo 'C:\Users\janaina\OneDrive\Documentos\UECE-RNP-2024-REF\intervalos-vazao\bbr\intervalos vazao bbr esmond data ac-ce 2024-2025.csv' processado via SVD: RMSE final = 0.000766 em 51 iterações.
Arquivo CSV 'C:\Users\janaina\OneDrive\Documentos\UECE-RNP-2024-REF\intervalos-vazao\bbr-imputed\svd\intervalos vazao bbr esmond data ac-ce 2024-2025.csv' gerado com sucesso!
Arquivo 'C:\Users\janaina\OneDrive\Documentos\UECE-RNP-2024-REF\intervalos-vazao\bbr\intervalos vazao bbr esmond data ac-rs 2024-2025.csv' processado via SVD: RMSE final = 0.000694 em 57 iterações.
Arquivo CSV 'C:\Users\janaina\OneDrive\Documentos\UECE-RNP-2024-REF\intervalos-vazao\bbr-imputed\svd\intervalos vazao bbr esmond data ac-rs 2024-2025.csv' gerado com sucesso!
Arquivo 'C:\Users\janaina\OneDrive\Documentos\UECE-RNP-2024-REF\intervalos-vazao\bbr\intervalos vazao bbr esmond data ac-sp 2024-2025.csv' processado via SVD: RMSE final = 0.000948 em 173 iterações.
Arquivo CS